# Session 2 · Words Defined by Other Words
*Understanding Language Models*

**In this session you will:** see how a machine turns words into points in space, map a vocabulary from the arts, and test the old structuralist idea that a word means what it means because of its *relationships* to other words.

**Time:** about 60–75 minutes.

### How to use this notebook
- This is a **Google Colab notebook**: a page that mixes reading with small pieces of code you can run.
- To run a grey code box, click it and press **Shift + Enter** (or click the ▶ button on its left).
- **Run the boxes in order, top to bottom.** If something breaks, go to *Runtime → Restart session* and start again from the top.
- You never have to *write* code. Where you see text inside quotation marks, like `"this"`, you can change the words and run the box again. That is the whole skill.
- Boxes marked **Setup** load the machinery. You can open them if you are curious, but you do not need to read them.

## The big idea

About a hundred years ago, the Swiss linguist Ferdinand de Saussure argued that a word has no meaning on its own. "Red" means something only because it is not "orange", "crimson" or "blue". Language, for him, was a **system of differences**: a web where each sign is defined by its position among the others.

Most of the twentieth century's thinking about meaning went another way, treating words mainly as labels that **refer** to things in the world. A useful name for this assumption is the **ladder of reference**: pointing at the world comes first, and everything else (style, genre, poetry, ideology) is built on top.

His argument is that language models turn the ladder upside down. They have never seen, heard or touched anything. They learn only how words sit next to other words. And yet something recognisable as meaning emerges. The technical name for this is an **embedding**: every word becomes a list of numbers, a point in a space where nearby points have related meanings.

Today we build a small map of that space.

## Setup

In [ ]:
#@title Setup: load a small embedding model (about a minute)
!pip -q install sentence-transformers
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

VOCAB = {
    "music":   ["jazz", "highlife", "drum", "melody", "rhythm", "saxophone", "choir", "song", "guitar", "concert"],
    "visual":  ["painting", "sculpture", "canvas", "portrait", "gallery", "brush", "colour", "photograph", "mural", "exhibition"],
    "writing": ["novel", "poem", "story", "author", "chapter", "metaphor", "library", "essay", "rhyme", "verse"],
    "food":    ["bread", "soup", "rice", "pepper", "kitchen", "fufu", "stew", "market", "fish", "banana"],
    "feeling": ["joy", "grief", "anger", "love", "fear", "hope", "shame", "pride", "longing", "calm"],
    "money":   ["funding", "grant", "budget", "salary", "sponsor", "invoice", "profit", "tax", "loan", "donor"],
}
ALL_WORDS = [w for group in VOCAB.values() for w in group]
_cache = {}

def vec(text):
    if text not in _cache:
        _cache[text] = emb_model.encode(text, normalize_embeddings=True)
    return _cache[text]

def similarity(a, b):
    return float(np.dot(vec(a), vec(b)))

def compare(pairs):
    rows = [(a, b, round(similarity(a, b), 2)) for a, b in pairs]
    return pd.DataFrame(rows, columns=["word A", "word B", "similarity"]).sort_values("similarity", ascending=False)

def neighbours(word, k=8, extra=()):
    pool = [w for w in ALL_WORDS + list(extra) if w != word]
    scores = sorted(((similarity(word, w), w) for w in pool), reverse=True)[:k]
    return pd.DataFrame([(w, round(s, 2)) for s, w in scores], columns=["nearest words", "similarity"])

def draw_map(groups):
    words, labels = [], []
    for g, ws in groups.items():
        words += ws; labels += [g] * len(ws)
    pts = PCA(n_components=2).fit_transform(np.array([vec(w) for w in words]))
    plt.figure(figsize=(10, 7))
    for g in groups:
        idx = [i for i, l in enumerate(labels) if l == g]
        plt.scatter(pts[idx, 0], pts[idx, 1], label=g, s=60)
        for i in idx:
            plt.annotate(words[i], pts[i], fontsize=10, xytext=(4, 3), textcoords="offset points")
    plt.legend(); plt.title("A map of words, drawn only from how they are used")
    plt.xticks([]); plt.yticks([]); plt.show()

def analogy(a, b, c, extra=()):
    """a is to b as c is to ... ?"""
    target = vec(b) - vec(a) + vec(c)
    target /= np.linalg.norm(target)
    pool = [w for w in ALL_WORDS + list(extra) if w not in (a, b, c)]
    best = sorted(((float(np.dot(target, vec(w))), w) for w in pool), reverse=True)[:5]
    print(f"{a} is to {b} as {c} is to ...")
    for s, w in best:
        print(f"   {w:<14} {s:.2f}")

print("Embedding model ready.")

## Part 1 · How close are two words?

Each word is now a point. **Similarity** runs from about 0 (unrelated) to 1 (nearly the same). Run the box, then add your own pairs.

In [ ]:
compare([
    ("painting", "canvas"),
    ("painting", "portrait"),
    ("painting", "soup"),
    ("jazz", "highlife"),
    ("grief", "longing"),
    ("grant", "donor"),
    ("poem", "invoice"),
])

**Try:** pairs that you think are close but the machine might not (words from your own practice, local terms, slang). Where does it agree with you and where does it not? Remember it only knows what it has read.

## Part 2 · Neighbourhoods

Which words live near a given word? The machine has never seen a drum or tasted a stew. Everything below comes from patterns of use.

In [ ]:
neighbours("rhythm")

In [ ]:
# Add your own words to the pool with extra=[...]
neighbours("unicorn", extra=["horse", "myth", "dragon", "legend", "fantasy", "forest"])

**Notice:** nobody has ever seen a unicorn, yet "unicorn" has a perfectly sensible neighbourhood. This is a direct challenge to the "grounding" debate. If meaning required a link to something we have perceived, "unicorn" would be meaningless. It is not. Its meaning comes from its place in a web of other words.

## Part 3 · Draw the map

Now we flatten the whole vocabulary down to two dimensions so we can look at it. Words were coloured by *us*, using the groups in the setup box. The *positions* come entirely from the machine.

In [ ]:
draw_map(VOCAB)

**Look for:**
- Do the groups form clusters without being told to?
- Which words sit between groups? (Is "market" closer to food or money? Is "verse" nearer to writing or music?)
- Which placements surprise you, and what might that say about the texts it learned from?

You can build your own map. Replace the words below with vocabulary from your field.

In [ ]:
my_groups = {
    "dance":   ["ballet", "choreography", "rehearsal", "stage", "costume", "adowa"],
    "cinema":  ["film", "director", "screen", "camera", "actor", "premiere"],
    "craft":   ["pottery", "weaving", "beads", "loom", "clay", "basket"],
}
draw_map(my_groups)

## Part 4 · Meaning as difference: analogies

If meaning is a set of relationships, we should be able to do something odd: take the *difference* between two words and apply it to a third. This is the famous "king − man + woman = queen" trick.

In [ ]:
analogy("painting", "painter", "poem", extra=["poet", "singer", "reader", "writer", "sculptor"])

In [ ]:
analogy("drum", "rhythm", "brush", extra=["stroke", "texture", "line", "shape"])

Sometimes it works beautifully, sometimes not at all. That is honest: the relationships are real but messy, and they come from *usage*, which is messier than any dictionary.

## Part 5 · Context changes everything

Single words are only half the story. Modern models read each word **in context**, so the same word gets a different position depending on the sentence around it. Compare:

In [ ]:
compare([
    ("She sat on the bank of the river.", "The water was cold at the edge of the stream."),
    ("She sat on the bank of the river.", "The bank refused our loan application."),
    ("The band played a set until midnight.", "The musicians performed late into the night."),
    ("The band played a set until midnight.", "She wore a gold band on her finger."),
])

This is closer to how language actually works for us: meaning lives in use, in the sentence, in the situation. Saussure's "system of differences" turns out to be something you can compute, and this is arguably why these machines work at all.

## Discussion

1. Does it unsettle you that a machine can place "grief" near "longing" without ever having felt either? Why, or why not?
2. One argument is that critics who insist machine text has "no meaning" are defending a picture of language where meaning must always come from a human mind. Where do *you* think meaning lives: in the speaker, the listener, the words, or between them?
3. Whose usage built this map? If most of the text came from English-language websites, whose meanings for "market", "funding" or "highlife" have been captured?

## Glossary
- **Embedding**: a word (or sentence) turned into a list of numbers, a point in space.
- **Similarity**: how close two points are.
- **Structuralism**: the tradition (Saussure, later Jakobson, Lévi-Strauss, Barthes) that treats meaning as a system of relationships.
- **Ladder of reference**: the assumption that pointing at the world is language's foundation.

## Going further
- Allison Parrish's talks and essays on poetry and word vectors.